# Buổi 3 - Notebook thực hành Detection baseline (YOLO)

Notebook này giúp bạn chạy Buổi 3 theo thứ tự:
1. Cài đặt môi trường cần thiết
2. Kiểm tra dữ liệu + split
3. Tạo `data/data.yaml`
4. Train YOLO baseline
5. Chạy inference và lưu kết quả
6. Tổng hợp checkpoint cho Buổi 4

> Gợi ý: chạy từng cell từ trên xuống để dễ debug.

In [ ]:
from pathlib import Path
import os
import sys
import json

# Chuẩn hóa project root để notebook chạy được cả khi mở từ docs/ hoặc từ root
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "docs" else cwd
os.chdir(project_root)

print("Project root:", project_root)
print("Python:", sys.version.split()[0])

## 1) Cài thư viện cần thiết

- Nếu bạn đã cài từ trước thì có thể bỏ qua cell install.
- Khuyến nghị dùng cùng môi trường với các buổi trước.

In [ ]:
import subprocess

RUN_INSTALL = False  # Đổi thành True nếu muốn cài toàn bộ requirements
AUTO_INSTALL_ULTRALYTICS = True  # Tự cài ultralytics nếu thiếu

if RUN_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

if sys.version_info >= (3, 13):
    print("Canh bao: Python >= 3.13 co the khong tuong thich on dinh voi torch/ultralytics.")
    print("Khuyen nghi dung Python 3.10 hoac 3.11 neu cai dat that bai.")

# Kiểm tra package tối thiểu
try:
    __import__("ultralytics")
    print("Package ultralytics da san sang.")
except Exception:
    print("Thieu package: ultralytics")
    if AUTO_INSTALL_ULTRALYTICS:
        print("Dang thu cai ultralytics vao dung kernel hien tai...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "ultralytics"])
        __import__("ultralytics")
        print("Da cai xong ultralytics. Neu van loi, hay Restart Kernel roi chay lai.")
    else:
        print("Hay bat AUTO_INSTALL_ULTRALYTICS=True hoac tu cai bang pip.")

## 2) Kiểm tra dữ liệu và split

Cell dưới kiểm tra nhanh các thư mục chính và các file split của Buổi 2.

In [ ]:
from collections import Counter

images_dir = Path("data/images/raw")
labels_dir = Path("data/labels/raw")
splits_dir = Path("data/splits")

required_files = [
    splits_dir / "train.txt",
    splits_dir / "val.txt",
    splits_dir / "test.txt",
]

print("images_dir exists:", images_dir.exists())
print("labels_dir exists:", labels_dir.exists())
print("splits_dir exists:", splits_dir.exists())

for f in required_files:
    print(f"{f}:", "OK" if f.exists() else "MISSING")

def read_split(path: Path):
    if not path.exists():
        return []
    return [ln.strip() for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]

split_map = {name: read_split(splits_dir / f"{name}.txt") for name in ["train", "val", "test"]}
for k, v in split_map.items():
    print(f"{k}: {len(v)} images")

# Kiểm tra nhanh số ảnh/nhãn thiếu
missing_image = Counter()
missing_label = Counter()
for split_name, image_list in split_map.items():
    for img_path in image_list:
        p = Path(img_path)
        if not p.is_absolute():
            p = project_root / p

        if not p.exists():
            missing_image[split_name] += 1
            continue

        label_path = labels_dir / f"{p.stem}.txt"
        if not label_path.exists():
            missing_label[split_name] += 1

print("Missing images:", dict(missing_image) if missing_image else "None")
print("Missing labels:", dict(missing_label) if missing_label else "None")

## 3) Tạo file `data/data.yaml`

File này dùng cho Ultralytics YOLO khi train/eval.

In [ ]:
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

data_yaml_path = data_dir / "data.yaml"

yaml_text = """path: .
train: data/splits/train.txt
val: data/splits/val.txt
test: data/splits/test.txt

names:
  0: license_plate
"""

data_yaml_path.write_text(yaml_text, encoding="utf-8")

print("Saved:", data_yaml_path)
print(data_yaml_path.read_text(encoding="utf-8"))

## 4) Train YOLO baseline (Buổi 3)

Bạn có thể chỉnh các tham số train ở cell dưới tùy theo GPU/CPU.

In [ ]:
import subprocess

# Ensure ultralytics is available in the active kernel env
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    print("Khong tim thay ultralytics. Dang cai dat tu dong...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "ultralytics"])
    from ultralytics import YOLO

# Cấu hình baseline
model_name = "yolov8n.pt"
epochs = 50
imgsz = 640
batch = 16
project = "experiments"
run_name = "yolo_buoi3_baseline"

if "data_yaml_path" not in globals() or not Path(data_yaml_path).exists():
    raise FileNotFoundError("Khong tim thay data_yaml_path. Hay chay cell tao data/data.yaml truoc.")

model = YOLO(model_name)

train_results = model.train(
    data=str(data_yaml_path),
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    project=project,
    name=run_name,
    exist_ok=True,
)

print("Train finished.")
print("Save dir:", train_results.save_dir)

## 5) Xem nhanh metrics sau train

Cell dưới đọc `results.csv` để xem các chỉ số cuối cùng của run.

In [ ]:
import csv

save_dir = Path(train_results.save_dir)
results_csv = save_dir / "results.csv"

metric_aliases = {
    "train/box_loss": "train_box_loss",
    "train/cls_loss": "train_cls_loss",
    "metrics/precision(B)": "precision",
    "metrics/recall(B)": "recall",
    "metrics/mAP50(B)": "mAP50",
    "metrics/mAP50-95(B)": "mAP50_95",
}

if results_csv.exists():
    with results_csv.open("r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    print("Số dòng metrics:", len(rows))

    selected_rows = []
    if rows:
        selected_indices = [0, 1, 2, max(0, len(rows) - 3), max(0, len(rows) - 2), len(rows) - 1]
        seen_indices = set()
        for idx in selected_indices:
            if idx in seen_indices:
                continue
            seen_indices.add(idx)
            row = rows[idx]
            selected_rows.append({"epoch": row.get("epoch", str(idx + 1))})
            selected_rows[-1].update({alias: row.get(col, "") for col, alias in metric_aliases.items()})

    print("\nBaseline metrics ở vài epoch đầu và cuối:")
    for row in selected_rows:
        print(row)

    if rows:
        first = selected_rows[0]
        last = selected_rows[-1]
        print("\nTóm tắt ban đầu -> cuối:")
        for key in ["train_box_loss", "train_cls_loss", "precision", "recall", "mAP50", "mAP50_95"]:
            print(f"- {key}: {first.get(key)} -> {last.get(key)}")
else:
    print("Không tìm thấy results.csv. Kiểm tra lại cell train.")

## 6) Chạy inference với checkpoint tốt nhất

Cell dưới:
- Lấy `best.pt` của run
- Copy về `weights/yolov8_license_plate.pt`
- Chạy script `scripts/run_infer.py` với `ocr-backend dummy` để tập trung đánh giá detection baseline.

In [ ]:
import shutil
import subprocess

weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)

best_pt = save_dir / "weights" / "best.pt"
target_pt = weights_dir / "yolov8_license_plate.pt"

if best_pt.exists():
    shutil.copy2(best_pt, target_pt)
    print("Copied:", best_pt, "->", target_pt)
else:
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {best_pt}")

output_json = Path("outputs/predictions_buoi3.json")
output_json.parent.mkdir(parents=True, exist_ok=True)

# Demo nhanh trong notebook: chạy vài ảnh trước, không quét toàn bộ dataset.
max_images = 20
cmd = [
    sys.executable,
    "scripts/run_infer.py",
    "--input-dir", "data/images/raw",
    "--output-json", str(output_json),
    "--detector-backend", "yolov8",
    "--detector-model", str(target_pt),
    "--ocr-backend", "dummy",
    "--max-images", str(max_images),
]

print("Running:", " ".join(cmd))
completed = subprocess.run(cmd, text=True, capture_output=True)

if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

completed.check_returncode()
print(f"Saved predictions: {output_json} (max_images={max_images})")

## 7) Kiểm tra nhanh output inference

In [ ]:
if output_json.exists():
    preds = json.loads(output_json.read_text(encoding="utf-8"))
    print("Num predictions:", len(preds))
    print("Sample rows:")
    for row in preds[:3]:
        print(row)
else:
    print("Chưa có output JSON. Hãy chạy cell inference trước.")

## 8) Phân tích lỗi sơ bộ detection

Cell dưới xem nhanh một vài ảnh inference, vẽ bbox dự đoán và bbox ground-truth nếu có nhãn YOLO. Mục tiêu là ghi nhận lỗi sơ bộ như: detect lệch, biển nhỏ, ảnh mờ, hoặc nhầm background.

In [ ]:
import math

import cv2
import matplotlib.pyplot as plt


def yolo_label_to_xyxy(label_path: Path, image_width: int, image_height: int):
    if not label_path.exists():
        return None

    lines = [ln.strip() for ln in label_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    if not lines:
        return None

    parts = lines[0].split()
    if len(parts) < 5:
        return None

    _, x_center, y_center, width, height = map(float, parts[:5])
    x1 = int((x_center - width / 2) * image_width)
    y1 = int((y_center - height / 2) * image_height)
    x2 = int((x_center + width / 2) * image_width)
    y2 = int((y_center + height / 2) * image_height)
    return [x1, y1, x2, y2]


def bbox_iou(box_a, box_b) -> float:
    if box_a is None or box_b is None:
        return 0.0

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union_area = area_a + area_b - inter_area
    return inter_area / union_area if union_area else 0.0


def classify_detection_error(iou: float, pred_box, image_width: int, image_height: int, blur_score: float) -> str:
    if pred_box is None:
        return "không detect"

    x1, y1, x2, y2 = pred_box
    pred_area_ratio = max(0, x2 - x1) * max(0, y2 - y1) / (image_width * image_height)

    if iou >= 0.5:
        if pred_area_ratio < 0.01:
            return "detect đúng nhưng biển nhỏ"
        if blur_score < 80:
            return "detect đúng, ảnh hơi mờ"
        return "detect ổn"
    if iou >= 0.1:
        return "detect lệch"
    return "nhầm background / sai vùng"


if "output_json" not in globals() or not output_json.exists():
    raise FileNotFoundError("Chưa có predictions. Hãy chạy cell inference trước.")

preds = json.loads(output_json.read_text(encoding="utf-8"))
review_preds = preds[: min(12, len(preds))]
if not review_preds:
    raise ValueError("File predictions đang rỗng, chưa có ảnh để phân tích lỗi.")

error_rows = []
cols = 3
rows_count = math.ceil(len(review_preds) / cols)
fig, axes = plt.subplots(rows_count, cols, figsize=(15, 4 * rows_count))
axes = list(axes.flatten()) if hasattr(axes, "flatten") else [axes]

for ax, pred in zip(axes, review_preds):
    image_path = Path(pred["source"])
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        ax.set_title(f"Không đọc được ảnh: {image_path}")
        ax.axis("off")
        continue

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_height, image_width = image_rgb.shape[:2]

    pred_box = pred.get("bbox_xyxy")
    label_path = Path("data/labels/raw") / f"{image_path.stem}.txt"
    gt_box = yolo_label_to_xyxy(label_path, image_width, image_height)
    iou = bbox_iou(pred_box, gt_box)
    blur_score = cv2.Laplacian(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
    error_type = classify_detection_error(iou, pred_box, image_width, image_height, blur_score)

    if gt_box:
        x1, y1, x2, y2 = gt_box
        cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)

    if pred_box:
        x1, y1, x2, y2 = map(int, pred_box)
        cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (255, 0, 0), 2)

    ax.imshow(image_rgb)
    ax.set_title(f"{image_path.name}\nIoU={iou:.2f} | {error_type}")
    ax.axis("off")

    error_rows.append(
        {
            "image_id": pred.get("image_id"),
            "iou": round(iou, 3),
            "confidence": pred.get("confidence"),
            "blur_score": round(float(blur_score), 1),
            "error_type": error_type,
        }
    )

for ax in axes[len(review_preds):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Ghi chú: xanh lá = ground-truth, đỏ = dự đoán.")
print("\nBảng phân tích lỗi sơ bộ:")
for row in error_rows:
    print(row)

## 9) Checklist nghiệm thu Buổi 3

- [ ] Có `data/data.yaml` hợp lệ
- [ ] Train YOLO chạy hoàn tất và có `best.pt`
- [ ] Có `outputs/predictions_buoi3.json`
- [ ] Ghi lại loss/metrics baseline ở vài epoch đầu và cuối (`train/box_loss`, `train/cls_loss`, `precision`, `recall`, `mAP50`, `mAP50-95`)
- [ ] Có phân tích lỗi sơ bộ: detect lệch, biển nhỏ, ảnh mờ, nhầm background

---

# Phụ lục học nhanh: hiểu YOLO baseline từ mất gốc

Phần này giúp bạn hiểu notebook theo cách trực quan hơn, nhưng vẫn giữ file nhẹ:

- Không nhúng ảnh nặng hoặc ảnh base64 vào notebook.
- Các sơ đồ và biểu đồ bên dưới được **vẽ bằng code khi chạy cell**.
- Nếu muốn nộp báo cáo, bạn có thể chạy cell rồi chụp hình kết quả, hoặc lưu riêng ra `docs/assets/`.

## Ý tưởng chính

Bài toán Buổi 3 là **object detection**:

$$
\text{Ảnh đầu vào} \rightarrow \text{YOLO} \rightarrow \text{bbox biển số } [x_1, y_1, x_2, y_2]
$$

Ở buổi này, ta mới kiểm tra khả năng **khoanh vùng biển số**, chưa đánh giá OCR đọc chữ thật.

In [ ]:
# Sơ đồ luồng hoạt động nhẹ: hình chỉ được tạo khi chạy cell này.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

steps = [
    ("1. Ảnh + nhãn", "data/images/raw\ndata/labels/raw"),
    ("2. Split", "train / val / test"),
    ("3. data.yaml", "Khai báo đường dẫn\nvà lớp license_plate"),
    ("4. Train YOLOv8n", "Học bbox từ nhãn"),
    ("5. best.pt", "Checkpoint tốt nhất"),
    ("6. Inference", "Dự đoán bbox\ntrên ảnh mới"),
    ("7. Đánh giá", "IoU, Precision,\nRecall, mAP"),
]

fig, ax = plt.subplots(figsize=(16, 4))
ax.set_xlim(0, len(steps))
ax.set_ylim(0, 1)
ax.axis("off")

for idx, (title, body) in enumerate(steps):
    x = idx + 0.08
    box = FancyBboxPatch(
        (x, 0.25),
        0.78,
        0.5,
        boxstyle="round,pad=0.03,rounding_size=0.04",
        linewidth=1.5,
        edgecolor="#4C78A8",
        facecolor="#EEF5FF",
    )
    ax.add_patch(box)
    ax.text(x + 0.39, 0.58, title, ha="center", va="center", fontsize=10, fontweight="bold")
    ax.text(x + 0.39, 0.42, body, ha="center", va="center", fontsize=9)
    if idx < len(steps) - 1:
        ax.annotate("", xy=(idx + 1.03, 0.5), xytext=(idx + 0.88, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))

ax.set_title("Luồng hoạt động YOLO baseline trong notebook Buổi 3", fontsize=14, fontweight="bold")
plt.show()

## Các thuật toán được dùng

### 1) YOLOv8n cho object detection

YOLO là mô hình phát hiện vật thể một giai đoạn. Hiểu đơn giản:

1. **Backbone**: nhìn ảnh và trích xuất đặc trưng như cạnh, góc, vùng sáng/tối, hình dạng biển số.
2. **Neck**: kết hợp đặc trưng ở nhiều kích thước để phát hiện cả biển số nhỏ và lớn.
3. **Head**: dự đoán bbox, lớp vật thể và confidence score.

Với bài này chỉ có một lớp:

$$
\text{class } 0 = \text{license\_plate}
$$

Mỗi dự đoán có dạng:

$$
\hat{y} = (x_1, y_1, x_2, y_2, c, p)
$$

Trong đó:

- $(x_1, y_1, x_2, y_2)$ là tọa độ bbox.
- $c$ là độ tin cậy có vật thể.
- $p$ là xác suất thuộc lớp `license_plate`.

### 2) Hàm loss khi train

Khi train, YOLO so sánh dự đoán với nhãn thật. Loss tổng quát có thể hiểu là:

$$
\mathcal{L} = \lambda_{box}\mathcal{L}_{box} + \lambda_{cls}\mathcal{L}_{cls} + \lambda_{dfl}\mathcal{L}_{dfl}
$$

Ý nghĩa:

- $\mathcal{L}_{box}$: phạt khi bbox dự đoán lệch bbox thật.
- $\mathcal{L}_{cls}$: phạt khi phân loại sai lớp.
- $\mathcal{L}_{dfl}$: giúp tọa độ bbox chính xác hơn.

Trong notebook, hai chỉ số dễ nhìn nhất là `train_box_loss` và `train_cls_loss`. Cả hai giảm là dấu hiệu mô hình đang học tốt hơn.

### 3) NMS - Non-Maximum Suppression

Một biển số có thể bị model khoanh nhiều bbox gần giống nhau. NMS giữ bbox tốt nhất và loại các bbox trùng lặp.

Quy tắc đơn giản:

1. Sắp xếp bbox theo confidence từ cao xuống thấp.
2. Chọn bbox có confidence cao nhất.
3. Loại các bbox khác nếu IoU với bbox đã chọn quá cao.
4. Lặp lại cho đến khi hết bbox.

Nhờ NMS, kết quả cuối cùng gọn hơn và ít bị nhiều khung chồng lên một biển số.

## Công thức toán học cần nhớ

### 1) IoU - Intersection over Union

IoU đo độ trùng giữa bbox dự đoán $B_p$ và bbox thật $B_g$:

$$
IoU = \frac{Area(B_p \cap B_g)}{Area(B_p \cup B_g)}
$$

Diễn giải:

- $IoU = 1$: hai khung trùng hoàn toàn.
- $IoU \approx 0$: hai khung gần như không trùng.
- Trong object detection, nếu IoU vượt một ngưỡng, ví dụ $0.5$, dự đoán thường được xem là đúng.

### 2) Precision

$$
Precision = \frac{TP}{TP + FP}
$$

Precision trả lời câu hỏi: **Trong các vùng model nói là biển số, bao nhiêu vùng thật sự đúng?**

- `TP`: dự đoán đúng.
- `FP`: dự đoán nhầm, ví dụ khoanh nhầm nền hoặc bộ phận xe.

### 3) Recall

$$
Recall = \frac{TP}{TP + FN}
$$

Recall trả lời câu hỏi: **Trong tất cả biển số thật, model tìm được bao nhiêu biển số?**

- `FN`: biển số thật nhưng model bỏ sót.

### 4) F1-score

$$
F1 = \frac{2 \times Precision \times Recall}{Precision + Recall}
$$

F1 cân bằng giữa precision và recall. Nếu một trong hai thấp, F1 cũng thấp.

### 5) AP và mAP

AP là diện tích dưới đường Precision-Recall:

$$
AP = \int_0^1 Precision(Recall)\,dRecall
$$

mAP là trung bình AP:

$$
mAP = \frac{1}{N}\sum_{i=1}^{N} AP_i
$$

Vì bài này chỉ có một lớp `license_plate`, mAP gần như chính là AP của lớp biển số.

### 6) mAP50 và mAP50-95

$$
mAP50 = AP_{IoU=0.50}
$$

$$
mAP50\text{-}95 = \frac{1}{10}\sum_{t \in \{0.50, 0.55, ..., 0.95\}} AP_{IoU=t}
$$

`mAP50-95` khó hơn `mAP50` vì bbox phải chính xác ở nhiều ngưỡng IoU khắt khe hơn.

In [ ]:
# Minh họa IoU bằng hình chữ nhật.
# Cell này không lưu ảnh vào notebook; ảnh chỉ hiện khi chạy.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

pred_box_demo = [1.8, 1.4, 5.7, 3.6]  # x1, y1, x2, y2
true_box_demo = [1.2, 1.0, 5.2, 3.2]


def demo_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union_area = area_a + area_b - inter_area
    return inter_area / union_area if union_area else 0.0

fig, ax = plt.subplots(figsize=(7, 5))

# Ground truth bbox
x1, y1, x2, y2 = true_box_demo
ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="green", linewidth=3, label="Ground truth"))

# Predicted bbox
x1, y1, x2, y2 = pred_box_demo
ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=3, label="Prediction"))

# Intersection area
ix1 = max(pred_box_demo[0], true_box_demo[0])
iy1 = max(pred_box_demo[1], true_box_demo[1])
ix2 = min(pred_box_demo[2], true_box_demo[2])
iy2 = min(pred_box_demo[3], true_box_demo[3])
ax.add_patch(Rectangle((ix1, iy1), ix2 - ix1, iy2 - iy1, color="orange", alpha=0.25, label="Intersection"))

ax.set_xlim(0, 7)
ax.set_ylim(0, 5)
ax.set_aspect("equal")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right")
ax.set_title(f"Minh họa IoU = {demo_iou(pred_box_demo, true_box_demo):.3f}")
ax.set_xlabel("Trục x trong ảnh")
ax.set_ylabel("Trục y trong ảnh")
plt.show()

In [ ]:
# Biểu đồ đọc nhanh metrics từ results.csv.
# Nếu chưa có results.csv, cell sẽ dùng số liệu đã in trong notebook để minh họa.
import csv
from pathlib import Path

import matplotlib.pyplot as plt

fallback_metrics = {
    "epoch": [1, 2, 3, 48, 49, 50],
    "train_box_loss": [0.72458, 0.74432, 0.74622, 0.42069, 0.41691, 0.40818],
    "train_cls_loss": [1.28634, 0.72422, 0.58730, 0.21538, 0.20939, 0.20274],
    "precision": [0.94090, 0.96957, 0.96209, 0.98757, 0.98663, 0.98852],
    "recall": [0.95620, 0.93804, 0.95043, 0.98474, 0.98507, 0.98515],
    "mAP50": [0.97145, 0.98515, 0.98146, 0.99384, 0.99365, 0.99382],
    "mAP50_95": [0.75854, 0.81246, 0.79291, 0.89942, 0.89578, 0.90098],
}

metric_aliases_for_plot = {
    "train/box_loss": "train_box_loss",
    "train/cls_loss": "train_cls_loss",
    "metrics/precision(B)": "precision",
    "metrics/recall(B)": "recall",
    "metrics/mAP50(B)": "mAP50",
    "metrics/mAP50-95(B)": "mAP50_95",
}

plot_data = fallback_metrics
possible_results_csv = None
if "save_dir" in globals():
    possible_results_csv = Path(save_dir) / "results.csv"

if possible_results_csv and possible_results_csv.exists():
    with possible_results_csv.open("r", encoding="utf-8") as f:
        rows_for_plot = list(csv.DictReader(f))
    plot_data = {"epoch": [int(float(row.get("epoch", idx + 1))) for idx, row in enumerate(rows_for_plot)]}
    for source_col, alias in metric_aliases_for_plot.items():
        plot_data[alias] = [float(row[source_col]) for row in rows_for_plot if row.get(source_col, "") != ""]
else:
    print("Không tìm thấy results.csv trong session hiện tại, dùng số liệu mẫu đã in trong notebook.")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(plot_data["epoch"], plot_data["train_box_loss"], marker="o", label="box_loss")
axes[0].plot(plot_data["epoch"], plot_data["train_cls_loss"], marker="o", label="cls_loss")
axes[0].set_title("Loss giảm nghĩa là model học tốt hơn")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.25)
axes[0].legend()

axes[1].plot(plot_data["epoch"], plot_data["precision"], marker="o", label="precision")
axes[1].plot(plot_data["epoch"], plot_data["recall"], marker="o", label="recall")
axes[1].plot(plot_data["epoch"], plot_data["mAP50"], marker="o", label="mAP50")
axes[1].plot(plot_data["epoch"], plot_data["mAP50_95"], marker="o", label="mAP50-95")
axes[1].set_title("Metrics tăng nghĩa là detect tốt hơn")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0.7, 1.02)
axes[1].grid(True, alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.show()

## Cách đọc kết quả trong báo cáo

Bạn có thể diễn giải ngắn gọn như sau:

> Sau 50 epoch, `train_box_loss` giảm từ 0.72458 xuống 0.40818, cho thấy mô hình dự đoán bbox sát nhãn hơn. `train_cls_loss` giảm từ 1.28634 xuống 0.20274, cho thấy mô hình học tốt lớp `license_plate`. Các chỉ số `precision`, `recall`, `mAP50` và `mAP50-95` đều tăng, đặc biệt `mAP50-95` tăng từ 0.75854 lên 0.90098, chứng tỏ chất lượng định vị bbox cải thiện rõ rệt ở nhiều ngưỡng IoU.

Lưu ý quan trọng:

- `mAP50` cao nghĩa là model thường khoanh đúng vùng biển số ở ngưỡng dễ hơn.
- `mAP50-95` cao mới cho thấy bbox khoanh **sát hơn**, vì chỉ số này khắt khe hơn.
- Phần inference trong notebook dùng `ocr-backend dummy`, nên text `51H12345` chỉ là giá trị giả để kiểm thử pipeline, chưa phải kết quả OCR thật.

Tài liệu học thêm:

- Ultralytics YOLO Detection: https://docs.ultralytics.com/tasks/detect/
- Ultralytics Performance Metrics: https://docs.ultralytics.com/guides/yolo-performance-metrics/
- COCO Detection Evaluation: https://cocodataset.org/#detection-eval

## Ý nghĩa các thuật toán và công thức trong dự án này

Các công thức ở trên không phải để học thuộc rời rạc. Trong dự án nhận diện biển số, chúng trả lời các câu hỏi rất cụ thể:

### 1) YOLO giúp gì cho toàn bộ pipeline?

Pipeline tổng thể của dự án là:

$$
\text{Ảnh xe} \rightarrow \text{Detect biển số} \rightarrow \text{Cắt vùng biển số} \rightarrow \text{OCR đọc chữ} \rightarrow \text{Hậu xử lý} \rightarrow \text{Kết quả cuối}
$$

Buổi 3 chỉ tập trung vào bước đầu:

$$
\text{Ảnh xe} \rightarrow \text{YOLO} \rightarrow \text{bbox biển số}
$$

Ý nghĩa trong dự án:

- Nếu YOLO khoanh đúng biển số, OCR ở bước sau nhận được ảnh crop sạch hơn và dễ đọc hơn.
- Nếu YOLO khoanh lệch, OCR có thể bị mất chữ, dính nền, hoặc đọc sai.
- Nếu YOLO bỏ sót biển số, toàn bộ pipeline thất bại vì không có vùng nào để OCR.

Vì vậy, detector tốt là nền móng cho toàn bộ hệ thống nhận diện biển số.

### 2) IoU dùng để biết khoanh vùng có sát không

Trong notebook, bbox xanh là nhãn đúng, bbox đỏ là dự đoán. IoU đo độ trùng giữa hai bbox đó:

$$
IoU = \frac{Area(B_p \cap B_g)}{Area(B_p \cup B_g)}
$$

Ý nghĩa trong dự án:

- IoU cao: vùng cắt biển số gần đúng, OCR có nhiều khả năng đọc tốt.
- IoU thấp: vùng cắt bị lệch, có thể mất ký tự hoặc lấy nhầm nền.

Cách đọc nhanh:

- $IoU \ge 0.5$: thường xem là detect được biển số.
- $IoU \ge 0.75$: bbox khá sát.
- $IoU \ge 0.9$: bbox rất sát, phù hợp cho bước OCR.

Trong phần phân tích lỗi của notebook, nhiều mẫu có IoU khoảng 0.9, nghĩa là detector đang khoanh khá sát trên nhóm ảnh demo.

### 3) Precision cho biết mô hình có hay khoanh nhầm không

$$
Precision = \frac{TP}{TP + FP}
$$

Ý nghĩa trong dự án:

- Precision cao: model ít khoanh nhầm những vùng không phải biển số.
- Precision thấp: model dễ nhầm logo, đèn xe, bảng hiệu, chữ trên xe, hoặc vùng nền thành biển số.

Với hệ thống nhận diện biển số, precision thấp sẽ làm OCR phải đọc nhiều crop sai, tạo ra kết quả rác.

Trong notebook:

$$
Precision: 0.94090 \rightarrow 0.98852
$$

Diễn giải: sau khi train, trong các vùng model dự đoán là biển số, tỷ lệ đúng tăng lên rất cao.

### 4) Recall cho biết mô hình có bỏ sót biển số không

$$
Recall = \frac{TP}{TP + FN}
$$

Ý nghĩa trong dự án:

- Recall cao: phần lớn biển số thật đều được tìm thấy.
- Recall thấp: nhiều ảnh có biển số nhưng model không phát hiện ra.

Trong bài toán biển số, recall rất quan trọng vì nếu detector bỏ sót, OCR không có cơ hội sửa sai.

Trong notebook:

$$
Recall: 0.95620 \rightarrow 0.98515
$$

Diễn giải: mô hình sau train bỏ sót ít biển số hơn.

### 5) mAP50 và mAP50-95 dùng để đánh giá tổng quát chất lượng detector

`mAP50` đánh giá AP ở ngưỡng IoU 0.50:

$$
mAP50 = AP_{IoU=0.50}
$$

`mAP50-95` đánh giá trung bình trên nhiều ngưỡng IoU:

$$
mAP50\text{-}95 = \frac{1}{10}\sum_{t \in \{0.50, 0.55, ..., 0.95\}} AP_{IoU=t}
$$

Ý nghĩa trong dự án:

- `mAP50` cao: model thường tìm đúng vị trí biển số ở mức tương đối.
- `mAP50-95` cao: model không chỉ tìm đúng mà còn khoanh sát hơn.

Với OCR, `mAP50-95` quan trọng hơn vì OCR cần crop sát biển số, không chỉ cần khoanh đại khái.

Trong notebook:

$$
mAP50: 0.97145 \rightarrow 0.99382
$$

$$
mAP50\text{-}95: 0.75854 \rightarrow 0.90098
$$

Diễn giải: detector không chỉ phát hiện đúng biển số mà chất lượng bbox cũng tốt hơn rõ rệt.

## Kết quả sẽ được đánh giá như thế nào?

Để đánh giá kết quả Buổi 3, ta không chỉ nhìn một con số. Nên đánh giá theo 4 lớp: dữ liệu, loss, metrics, và kiểm tra hình ảnh.

### 1) Đánh giá dữ liệu đầu vào

Trước khi tin kết quả train, cần kiểm tra dữ liệu:

- Có đủ ảnh trong `train`, `val`, `test` không?
- Ảnh có nhãn YOLO tương ứng không?
- Có ảnh lỗi, thiếu nhãn, hoặc nhãn sai không?

Trong notebook, cell kiểm tra split giúp trả lời các câu hỏi này. Nếu dữ liệu sai, metrics cao/thấp đều không đáng tin.

### 2) Đánh giá quá trình học bằng loss

Ta xem các loss theo epoch:

- `train_box_loss`: giảm nghĩa là bbox dự đoán ngày càng sát bbox thật.
- `train_cls_loss`: giảm nghĩa là model phân biệt lớp `license_plate` tốt hơn.
- Nếu loss giảm đều và không dao động bất thường, quá trình học tương đối ổn.

Kết quả trong notebook:

$$
train\_box\_loss: 0.72458 \rightarrow 0.40818
$$

$$
train\_cls\_loss: 1.28634 \rightarrow 0.20274
$$

Đánh giá: loss giảm rõ, mô hình có học được đặc trưng biển số.

### 3) Đánh giá detection bằng metrics

Các chỉ số chính cần báo cáo:

- `Precision`: kiểm tra mô hình có khoanh nhầm không.
- `Recall`: kiểm tra mô hình có bỏ sót không.
- `mAP50`: đánh giá khả năng phát hiện đúng ở ngưỡng IoU 0.5.
- `mAP50-95`: đánh giá chất lượng bbox ở nhiều ngưỡng khó hơn.

Kết quả cuối notebook:

$$
Precision = 0.98852
$$

$$
Recall = 0.98515
$$

$$
mAP50 = 0.99382
$$

$$
mAP50\text{-}95 = 0.90098
$$

Đánh giá: đây là kết quả tốt cho detector baseline. `mAP50` rất cao cho thấy model phát hiện đúng vị trí biển số ở ngưỡng cơ bản. `mAP50-95` đạt khoảng 0.901 cho thấy bbox tương đối sát, có lợi cho bước OCR sau này.

### 4) Đánh giá trực quan bằng hình bbox

Metrics cao vẫn cần kiểm tra hình ảnh vì số liệu có thể che giấu lỗi cụ thể. Notebook vẽ:

- Xanh lá: bbox ground truth.
- Đỏ: bbox model dự đoán.
- IoU: độ trùng giữa hai bbox.

Cách đánh giá bằng mắt:

- Đỏ gần trùng xanh: detect tốt.
- Đỏ lệch một phần: có thể OCR mất ký tự.
- Đỏ quá rộng: OCR có thể đọc thêm nền hoặc chữ khác.
- Không có đỏ: model bỏ sót biển số.
- Đỏ ở vùng không phải biển số: model nhầm background.

Trong notebook, 12 mẫu kiểm tra nhanh đều được phân loại là `detect ổn`, nhiều IoU trên 0.9. Điều này ủng hộ kết luận rằng baseline hoạt động tốt trên mẫu demo.

### 5) Kết luận nên viết trong báo cáo

Có thể viết như sau:

> Mô hình YOLOv8n baseline cho bài toán phát hiện biển số đạt kết quả tốt sau 50 epoch. Loss giảm rõ rệt, precision đạt 0.98852, recall đạt 0.98515, mAP50 đạt 0.99382 và mAP50-95 đạt 0.90098. Điều này cho thấy mô hình ít khoanh nhầm, ít bỏ sót và bbox dự đoán khá sát với nhãn thật. Kiểm tra trực quan bằng bbox xanh/đỏ cũng cho thấy nhiều mẫu có IoU cao, phù hợp để dùng làm detector đầu vào cho bước OCR ở Buổi 4.

### 6) Giới hạn cần ghi rõ

Kết quả Buổi 3 mới đánh giá **detection**, chưa đánh giá OCR thật.

Trong cell inference, notebook dùng:

```bash
--ocr-backend dummy
```

Vì vậy text `51H12345` chỉ là giá trị giả để kiểm tra pipeline. Muốn đánh giá nhận diện biển số hoàn chỉnh, cần dùng OCR thật như EasyOCR hoặc TrOCR và báo cáo thêm:

- `CER`: lỗi theo ký tự.
- `WER`: lỗi theo token/từ.
- `plate_accuracy`: tỷ lệ đọc đúng toàn bộ biển số.
- `mean_latency_ms`: thời gian xử lý trung bình.

---

## File code liên quan đến notebook Buổi 3

Notebook này dùng để thực hành train và đánh giá baseline detection YOLO. Logic chính nằm trong các file sau:

### 1) Chuẩn bị dữ liệu YOLO

- [`../data/data.yaml`](../data/data.yaml): file cấu hình dataset cho Ultralytics YOLO, gồm đường dẫn `train`, `val`, `test` và tên lớp `license_plate`.
- [`../scripts/split_dataset.py`](../scripts/split_dataset.py): tạo split train/val/test nếu cần chạy lại từ đầu.
- [`../scripts/build_manifest.py`](../scripts/build_manifest.py): tạo manifest dữ liệu ảnh/nhãn.

### 2) Train detector

- [`../scripts/train_detector.py`](../scripts/train_detector.py): script train detector YOLO theo cấu hình dòng lệnh.
- [`../configs/deepsolo/README.md`](../configs/deepsolo/README.md): ghi chú cấu hình DeepSolo nếu muốn so sánh với hướng detector/text spotting khác.
- [`../configs/trocr/README.md`](../configs/trocr/README.md): ghi chú cấu hình TrOCR cho bước OCR ở buổi sau.

### 3) Detector YOLO trong pipeline

- [`../src/detector/base.py`](../src/detector/base.py): interface/base type cho detector.
- [`../src/detector/yolov8_detector.py`](../src/detector/yolov8_detector.py): wrapper YOLOv8, nhận ảnh và trả về bbox biển số.
- [`../src/utils/types.py`](../src/utils/types.py): định nghĩa kiểu dữ liệu như `Detection`, `FrameData`, `PipelineResult`.

### 4) Chạy inference sau khi train

- [`../scripts/run_infer.py`](../scripts/run_infer.py): entrypoint chạy inference từ terminal/notebook.
- [`../src/app/cli.py`](../src/app/cli.py): parse argument và kết nối các thành phần detector/OCR/pipeline.
- [`../src/pipeline/infer_plate_pipeline.py`](../src/pipeline/infer_plate_pipeline.py): luồng chính `detect -> crop -> preprocess -> OCR -> postprocess -> output`.
- [`../src/io/readers.py`](../src/io/readers.py): đọc ảnh/frame đầu vào.

### 5) OCR giả và hậu xử lý trong demo Buổi 3

- [`../src/ocr/base.py`](../src/ocr/base.py): có `DummyOcr`, tạo text giả `51H12345` để test pipeline detection.
- [`../src/preprocess/ops.py`](../src/preprocess/ops.py): crop và preprocess ảnh biển số.
- [`../src/postprocess/plate_rules.py`](../src/postprocess/plate_rules.py): chuẩn hóa và sửa lỗi OCR phổ biến.

### 6) Đánh giá detection/OCR ở các bước sau

- [`../scripts/eval_pipeline.py`](../scripts/eval_pipeline.py): chạy đánh giá pipeline khi có prediction và ground truth.
- [`../src/eval/metrics_plate.py`](../src/eval/metrics_plate.py): tính các metric text như CER, WER, plate accuracy.

### 7) Nên đọc theo thứ tự

1. [`../data/data.yaml`](../data/data.yaml) để hiểu YOLO lấy dữ liệu ở đâu.
2. [`../src/detector/yolov8_detector.py`](../src/detector/yolov8_detector.py) để hiểu bbox được dự đoán ra sao.
3. [`../src/pipeline/infer_plate_pipeline.py`](../src/pipeline/infer_plate_pipeline.py) để hiểu luồng inference đầy đủ.
4. [`../scripts/run_infer.py`](../scripts/run_infer.py) và [`../src/app/cli.py`](../src/app/cli.py) để hiểu lệnh notebook gọi ra hoạt động thế nào.